# Notebook 02_00 — Feature Rankings (compute once, reused by all FS method notebooks)

Training set has **3,397,858 rows**. Running RFE / SHAP directly on this is extremely slow
(this is the root cause of the original `02_feature_selection.ipynb` hanging/crashing).

**Fix**: compute all 6 rankings on a **stratified subsample** of the training set
(`RANKING_SAMPLE_SIZE = 100,000`, same class proportions as full train).
This only affects *which features are chosen* — the actual classifier training in
every downstream notebook still uses the full resampled training set.

Output: `models/fs_rankings.pkl` — a dict of `{method_name: ranked_feature_indices}`.

In [ ]:
import sys; sys.path.insert(0, '/home/hanh/UAV_/notebook')
from common import *

from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import chi2, RFE
from sklearn.tree import DecisionTreeClassifier
import shap

print('Imports OK')

In [ ]:
d = load_data()
X_train, y_train = d['X_train'], d['y_train']
FEATURE_NAMES = d['feature_names']

print(f'Full train shape: {X_train.shape}')

X_sub, y_sub = stratified_subsample(X_train, y_train, RANKING_SAMPLE_SIZE, seed=42)
print(f'Subsample shape : {X_sub.shape}  (used for ranking computation only)')
print(f'Subsample class counts:\n{pd.Series(y_sub).value_counts().sort_index()}')

## Ranking functions

In [ ]:
def rank_correlation(X, y):
    corr = np.array([abs(np.corrcoef(X[:, i], y)[0, 1]) for i in range(X.shape[1])])
    return np.argsort(np.nan_to_num(corr))[::-1]


def rank_chisquare(X, y):
    X_nn = MinMaxScaler().fit_transform(X)
    scores, _ = chi2(X_nn, y)
    return np.argsort(np.nan_to_num(scores))[::-1]


def rank_xgb_importance(X, y, seed=42):
    model = xgb.XGBClassifier(n_estimators=100, max_depth=6, random_state=seed,
                               n_jobs=N_JOBS, verbosity=0, eval_metric='mlogloss')
    model.fit(X, y)
    return np.argsort(model.feature_importances_)[::-1]


def rank_shap(X, y, seed=42):
    n_bg = min(500, len(X))
    idx = np.random.default_rng(seed).choice(len(X), n_bg, replace=False)
    model = xgb.XGBClassifier(n_estimators=100, max_depth=6, random_state=seed,
                               n_jobs=N_JOBS, verbosity=0, eval_metric='mlogloss')
    model.fit(X, y)
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X[idx])
    if isinstance(shap_values, list):
        mean_abs = np.mean([np.abs(sv).mean(axis=0) for sv in shap_values], axis=0)
    else:
        mean_abs = np.abs(shap_values).mean(axis=0)
    return np.argsort(mean_abs)[::-1]


def rank_rfe(X, y, seed=42):
    estimator = DecisionTreeClassifier(random_state=seed, max_depth=10)
    rfe = RFE(estimator=estimator, n_features_to_select=1, step=1)
    rfe.fit(X, y)
    return np.argsort(rfe.ranking_)


def rank_consensus(ranks_dict):
    n_feat = len(next(iter(ranks_dict.values())))
    avg_rank = np.zeros(n_feat)
    for ranked in ranks_dict.values():
        rank_array = np.empty(n_feat, dtype=int)
        for position, feat_idx in enumerate(ranked):
            rank_array[feat_idx] = position
        avg_rank += rank_array
    avg_rank /= len(ranks_dict)
    return np.argsort(avg_rank)


print('Ranking functions defined.')

## Compute all rankings (resumable: skip methods already saved)

In [ ]:
RANKINGS_PATH = f'{MODELS_DIR}fs_rankings.pkl'

if os.path.exists(RANKINGS_PATH):
    with open(RANKINGS_PATH, 'rb') as f:
        RANKINGS = pickle.load(f)
    print(f'Loaded existing rankings: {list(RANKINGS.keys())}')
else:
    RANKINGS = {}

RANK_FUNCS = {
    'Correlation'   : rank_correlation,
    'ChiSquare'     : rank_chisquare,
    'XGB_Importance': rank_xgb_importance,
    'SHAP'          : rank_shap,
    'RFE'           : rank_rfe,
}

for name, fn in RANK_FUNCS.items():
    if name in RANKINGS:
        print(f'{name}: already computed, skipping.')
        continue
    t0 = time.time()
    RANKINGS[name] = fn(X_sub, y_sub)
    print(f'{name}: done in {time.time()-t0:.1f}s')
    # Save after each method (resumable if this notebook itself crashes)
    with open(RANKINGS_PATH, 'wb') as f:
        pickle.dump(RANKINGS, f)

if 'Consensus' not in RANKINGS:
    RANKINGS['Consensus'] = rank_consensus(
        {k: v for k, v in RANKINGS.items() if k != 'Consensus'}
    )
    with open(RANKINGS_PATH, 'wb') as f:
        pickle.dump(RANKINGS, f)
    print('Consensus: done')

print(f'\nSaved all rankings -> {RANKINGS_PATH}')

## Inspect Top-16 features per method

In [ ]:
for method, ranked in RANKINGS.items():
    top = [FEATURE_NAMES[i] for i in ranked[:16]]
    print(f'{method:<16}: {top}')